# Chapter 34 — Performance, memory, pagination, caching, and scalability

**Book role:** Professional practice  
**Suggested book space:** about 8 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Identify the main performance bottlenecks in an analytics GUI.
- Choose limits, pagination, chunked reads, and caching deliberately.
- Avoid redundant widget creation and repeated recomputation.
- Measure before optimizing.

## 1. Measure the whole path

A slow table can be caused by connection latency, query execution, network transfer, DataFrame construction, cleaning, aggregation, formatting, or inserting thousands of Treeview items. Time each stage before optimizing. Report both wall-clock duration and row counts. A faster query that returns the wrong grain or fewer records is not an improvement.

Use realistic test volumes and cold/warm runs. Avoid measuring only a pure transformation on 100 rows and assuming it will scale to one million. Performance requirements should define data size, target response time, acceptable freshness, and hardware assumptions. For a student project, a clear measurement table and reasoned trade-offs are stronger than arbitrary optimization tricks.

### Worked example 1: Time the stages of a pipeline

Use the same dataset and environment for comparisons. Timing varies with storage cache, hardware, and run order, so do not treat one run as a benchmark.

In [ ]:
from time import perf_counter
import pandas as pd

start = perf_counter()
frame = pd.read_csv("data/sample_sales.csv", usecols=["order_id", "order_date", "region", "quantity", "unit_price", "unit_cost", "status"])
read_seconds = perf_counter() - start

start = perf_counter()
frame = frame.loc[frame["status"].eq("Paid")].copy()
frame["revenue"] = frame["quantity"] * frame["unit_price"]
frame["profit"] = frame["quantity"] * (frame["unit_price"] - frame["unit_cost"])
transform_seconds = perf_counter() - start
print({"rows_after_filter": len(frame), "read_seconds": round(read_seconds, 4), "transform_seconds": round(transform_seconds, 4)})

## 2. Control memory and amount of visible data

A DataFrame needs memory beyond the raw CSV file size; text objects and intermediate copies can multiply usage. Select required columns during import where possible, use appropriate dtypes, remove unused copies, and use chunked ingestion for data too large to fit comfortably. If downstream analysis needs all rows, think about streaming aggregation or database-side summaries rather than loading everything into the GUI.

Treeview should display a bounded set of rows with page controls. The table’s page size and the query limit are separate design choices: one limits UI objects, the other can limit data fetched. For a local DataFrame, pagination can slice sorted filtered rows; for a database, let the repository query one page and separately retrieve a safe total count if necessary.

### Worked example 2: Slice a page from a filtered DataFrame

The first output shows a full page and the second a partial final page. For database-backed paging, add deterministic ordering and push the limit/offset or keyset condition into the query.

In [ ]:
import pandas as pd

def dataframe_page(frame: pd.DataFrame, page: int, page_size: int = 200) -> pd.DataFrame:
    if page < 0:
        raise ValueError("page must be zero-based and non-negative")
    if not 1 <= page_size <= 2000:
        raise ValueError("page_size must be between 1 and 2000")
    start = page * page_size
    return frame.iloc[start:start + page_size].copy()

sample = pd.DataFrame({"value": range(1050)})
print(len(dataframe_page(sample, page=4, page_size=200)))
print(len(dataframe_page(sample, page=5, page_size=200)))

## 3. Caching and invalidation

Caching can speed repeated aggregations, but only if the cache key includes every input that changes the result: filter criteria, data version, date range, permissions, and relevant calculation settings. Invalidate or refresh the cache after a write, a source refresh, or a schema/rule change. Show timestamps for stale results and never use cached data as a silent substitute for a fresh result in a context where freshness matters.

Avoid needless UI churn: update the existing chart canvas, reuse table column definitions, batch row insertions, and debounce frequent control events. Batch updates and chunked work are usually more effective than micro-optimizing a string expression before measuring the full application.

## Guided lab: Lab: performance budget

Measure load, normalize, filter, aggregate, render, and export stages for 1,000, 50,000, and (where hardware permits) 250,000 rows. Define a page-size limit and an explicit large-dataset strategy. Record peak memory if tooling is available. Add a cache only after measuring repeated work and write down its cache key and invalidation conditions.

## Review questions and practice

1. Why can CSV file size underestimate required memory?
2. What distinction exists between a UI page-size limit and a query limit?
3. Which values belong in a cache key?
4. Why should performance be measured before optimizing?

## Further reading

- pandas scaling guide: https://pandas.pydata.org/docs/user_guide/scale.html
- Python profiling: https://docs.python.org/3/library/profile.html

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.